# 🔄 Notebook 4: Preprocessing & Handling Class Imbalance

Before training, we need to:
1. Handle the severe class imbalance (0.13% fraud)
2. Scale numerical features for Logistic Regression
3. Train/test split with stratification
4. Apply SMOTE oversampling to the training set only


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import sys
sys.path.insert(0, '..')
from src.data_loader import load_data
from src.feature_engineering import engineer_features, get_feature_names
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#444',
})

DATA_PATH = '../data/PS_20174392719_1491204439457_log.csv'
df = load_data(DATA_PATH, sample_frac=0.2)
df_fe = engineer_features(df)

feature_cols = get_feature_names(df_fe)
feature_cols = [c for c in feature_cols if c in df_fe.columns]
X = df_fe[feature_cols].fillna(0)
y = df_fe['isFraud']

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution: {Counter(y)}")


In [ ]:
# ── Train-Test Split ──────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train/Test Split (stratified):")
print(f"  Train: {len(X_train):,} rows | Fraud: {y_train.sum():,} ({y_train.mean()*100:.3f}%)")
print(f"  Test:  {len(X_test):,} rows  | Fraud: {y_test.sum():,} ({y_test.mean()*100:.3f}%)")
print("  ✓ Stratification preserved fraud rate in both splits")


In [ ]:
# ── SMOTE Oversampling ─────────────────────────────────────────────────────
# CRITICAL: SMOTE is applied ONLY to training data, never to test data
print("Applying SMOTE to training set...")
sm = SMOTE(sampling_strategy=0.1, random_state=42, n_jobs=-1)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print(f"\nBefore SMOTE: {Counter(y_train)}")
print(f"After SMOTE:  {Counter(y_train_sm)}")
print(f"  New fraud rate: {y_train_sm.mean()*100:.2f}%")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Effect of SMOTE Oversampling on Training Set', fontsize=14, color='white')

for ax, (title, y_data) in zip(axes, [('Before SMOTE', y_train), ('After SMOTE', y_train_sm)]):
    counts = pd.Series(y_data).value_counts()
    ax.bar(['Non-Fraud', 'Fraud'], counts.values, color=['#4E79A7', '#E15759'], edgecolor='white')
    ax.set_title(title, color='white')
    ax.set_ylabel('Count', color='white')
    for i, v in enumerate(counts.values):
        ax.text(i, v * 1.01, f'{v:,}', ha='center', color='white', fontsize=10)

plt.tight_layout()
plt.savefig('../visuals/04_smote_effect.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


In [ ]:
# ── Feature Scaling ───────────────────────────────────────────────────────
# Only needed for Logistic Regression — tree models don't need scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

print("StandardScaler applied:")
print(f"  Train mean (sample): {X_train_scaled[:, 0].mean():.4f} (should be ~0)")
print(f"  Train std (sample):  {X_train_scaled[:, 0].std():.4f} (should be ~1)")
print("  ✓ Scaler fit on SMOTE-augmented training data only")
print("  ✓ Test data scaled using training statistics (no data leakage)")

import pickle, os
os.makedirs('../models', exist_ok=True)
pickle.dump(scaler, open('../models/scaler.pkl', 'wb'))
print("\n✅ Scaler saved to ../models/scaler.pkl")
print("\n✅ Preprocessing complete — data ready for modeling!")


## ✅ Preprocessing Summary

| Step | Decision | Reason |
|---|---|---|
| Train/Test split | 80/20, stratified | Preserve class ratio |
| Class imbalance | SMOTE (10% fraud rate) | Synthetic minority oversampling |
| Feature scaling | StandardScaler | Required for Logistic Regression |
| Test set | Never touched by SMOTE or fit transforms | Prevents data leakage |
| Missing values | Fill with 0 | Feature engineering removes most NaN |

**Key principle:** All preprocessing steps are fit on training data and applied to test data — no leakage.

➡️ **Next:** Notebook 5 — Model Training
